In [1]:
import warnings
import os
warnings.simplefilter(action='ignore')
os.environ["PYTHONWARNINGS"] = "ignore"

In [2]:
#parameters

### USER EDIT start
esm_file='/g/data/zv30/non-cmip/ACCESS-CM3/cm3-run-03-06-2026/cm3-datastore/cm3-datastore.json'
# esm_file= os.path.join(run_dir, 'cm3-demo-datastore/cm3-virtual-datastore.json')
dpi=300

# papermill settings. *No need to modify these if running interactively.*
papermill = False                      # `cwd` and `nbname` will be populated by papermill.
cwd = None                             # current working directory
nbname = None                          # notebook name


In [3]:
# Parameters
esm_file = "/g/data/zv30/non-cmip/ACCESS-CM3/cm3-run-27-07-2026-PD-control/cm3-datastore/cm3-datastore.json"
papermill = True
cwd = "/g/data/tm70/cyb561/repos/get_cm3website-gogo/access-cm3-paper-1/notebooks/polished-python/mkfigs_output_cm3-run-27-07-2026-PD-control/"
nbname = "flux-conservation.ipynb"


In [4]:
import os  # always: the old parameters cell imported it, and notebooks use os.* (TOA_CRE does)
if not papermill:
    import nci_ipynb  # requires conda/analysis3-26.03 or later
    cwd = nci_ipynb.dir()
    nbname = nci_ipynb.name()
    os.chdir(cwd)
import mkfigs_bootstrap  # noqa: adds external/access-model-mkfigs/src to sys.path (stop-gap)
from mkfigs import MkmdWriter

import matplotlib as mpl
%matplotlib inline
mpl.rcParams['figure.dpi'] = dpi

mkmd = MkmdWriter(esm_file, nbname, str(cwd), pm=papermill)


In [5]:
import xarray as xr
import cf_xarray as cfxr
import intake
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
from distributed import Client
import numpy as np
import dask.array as da
import iris
import h5py

In [6]:
client = Client(threads_per_worker=1)
print(client.dashboard_link)

http://127.0.0.1:8787/status


## Load datastore and datasets

In [7]:
#datastore_path = "/g/data/ol01/access-om3-output/access-om3-025/MC_25km_jra_ryf-1.0-beta/experiment_datastore.json"
COLUMNS_WITH_ITERABLES = [
        "variable",
        "variable_long_name",
        "variable_standard_name",
        "variable_cell_methods",
        "variable_units"
]

datastore = intake.open_esm_datastore(
    esm_file,
    columns_with_iterables=COLUMNS_WITH_ITERABLES
)

## Load areas

In [8]:
wet = xr.load_dataset(datastore.search(variable="wet").df.path.iloc[0]).drop_dims('time')
areacello = xr.load_dataset(datastore.search(variable="areacello").df.path.iloc[0]).drop_dims('time')

areacello = (areacello.areacello * (wet.wet == 1.0))
ocn_area = areacello.sum().data

In [9]:
EARTH_RADIUS = 6371229.0
nlon, nlat = 192, 144
dx = 360 / nlon
dy = 180 / nlat

element_lat = (np.arange(nlat) + 0.5) * dy  - 90
element_lat = element_lat[:, None] * np.ones(nlon)[None, :]
element_lon = (np.arange(nlon) + 0.5) * dx

pi_over_180 = np.pi / 180
element_areas = dx * pi_over_180 * (
  np.sin((element_lat + 0.5 * dy) * pi_over_180) - np.sin((element_lat - 0.5 * dy) * pi_over_180)
)

areacella = element_areas * EARTH_RADIUS**2
areacella = xr.DataArray(areacella, coords=dict(lat=element_lat[:, 0], lon=element_lon), dims=('lat', 'lon'))

In [10]:
earth_area = areacella.data.sum()

In [11]:
ancil_dir = '/scratch/tm70/kr4383/cylc-run/ancil-gen-22-04-2026/share/data/n96e_mom025_20260316'
land_frac = iris.load_cube(os.path.join(ancil_dir, 'qrparm.landfrac')).data
vsat = iris.load_cube(os.path.join(ancil_dir, 'qrparm.soil'), 'soil_porosity').data
ocn_frac = 1 - land_frac

## Water flux error

### Ocean data

In [12]:
%%time
xr_kw = dict(chunks={"yh": -1, "xh": -1}, decode_timedelta=True)

ocn_river = datastore.search(variable="friver", frequency="1mon").to_dask(xarray_open_kwargs=xr_kw).friver
ocn_evap = datastore.search(variable="evs", frequency="1mon").to_dask(xarray_open_kwargs=xr_kw).evs
ocn_snow = datastore.search(variable="prsn", frequency="1mon").to_dask(xarray_open_kwargs=xr_kw).prsn
ocn_rain = datastore.search(variable="prlq", frequency="1mon").to_dask(xarray_open_kwargs=xr_kw).prlq
ocn_iceberg = datastore.search(variable="ficeberg", frequency="1mon").to_dask(xarray_open_kwargs=xr_kw).ficeberg
ocn_melt = datastore.search(variable="fsitherm", frequency="1mon").to_dask(xarray_open_kwargs=xr_kw).fsitherm
ocn_total = datastore.search(variable="wfo", frequency="1mon").to_dask(xarray_open_kwargs=xr_kw).wfo

CPU times: user 9.94 s, sys: 6.47 s, total: 16.4 s
Wall time: 24.7 s


In [13]:
ocn_river_avg = ocn_river.weighted(areacello.fillna(0)).sum(dim=('yh', 'xh')).compute() / ocn_area
ocn_evap_avg = ocn_evap.weighted(areacello.fillna(0)).sum(dim=('yh', 'xh')).compute() / ocn_area
ocn_snow_avg = ocn_snow.weighted(areacello.fillna(0)).sum(dim=('yh', 'xh')).compute() / ocn_area
ocn_rain_avg = ocn_rain.weighted(areacello.fillna(0)).sum(dim=('yh', 'xh')).compute() / ocn_area
ocn_iceberg_avg = ocn_iceberg.weighted(areacello.fillna(0)).sum(dim=('yh', 'xh')).compute() / ocn_area
ocn_melt_avg = ocn_melt.weighted(areacello.fillna(0)).sum(dim=('yh', 'xh')).compute() / ocn_area
ocn_total_avg = ocn_total.weighted(areacello.fillna(0)).sum(dim=('yh', 'xh')).compute() / ocn_area

### Atmosphere data

In [14]:
atm_fps = datastore.search(variable="fld_s26i004", frequency="1mon").df.sort_values(by='start_date').path.values
atm_ds_list = [h5py.File(fp, "r") for fp in atm_fps]

In [15]:
atm_river = xr.DataArray(da.concatenate([da.from_array(ds["/fld_s26i004"]) for ds in atm_ds_list]), dims=('time', 'lat', 'lon'))
atm_evap = xr.DataArray(da.concatenate([da.from_array(ds["/fld_s03i232"]) for ds in atm_ds_list]), dims=('time', 'lat', 'lon'))
atm_rain = xr.DataArray(da.concatenate([da.from_array(ds["/fld_s05i214"]) for ds in atm_ds_list]), dims=('time', 'lat', 'lon'))
atm_snow = xr.DataArray(da.concatenate([da.from_array(ds["/fld_s05i215"]) for ds in atm_ds_list]), dims=('time', 'lat', 'lon'))
atm_sublim = xr.DataArray(da.concatenate([da.from_array(ds["/fld_s03i298"]) for ds in atm_ds_list]), dims=('time', 'lat', 'lon'))

In [16]:
atm_river_avg = atm_river.weighted(areacella).sum(dim=("lat", "lon")).compute() / ocn_area
atm_evap_avg = atm_evap.weighted(ocn_frac * areacella).sum(dim=("lat", "lon")).compute() / ocn_area
atm_rain_avg = atm_rain.weighted(ocn_frac * areacella).sum(dim=("lat", "lon")).compute() / ocn_area
atm_snow_avg = atm_snow.weighted(ocn_frac * areacella).sum(dim=("lat", "lon")).compute() / ocn_area
atm_sublim_avg = atm_sublim.weighted(ocn_frac * areacella).sum(dim=("lat", "lon")).compute() / ocn_area

### Sea-ice data

In [17]:
ice_fps = sorted(datastore.search(variable="snow_ai_m", frequency="1mon").df.path.values)
ice_ds_list = [h5py.File(fp, "r") for fp in ice_fps]

In [18]:
ice_snow = xr.DataArray(da.concatenate([da.from_array(ds["/snow_ai_m"]) for ds in ice_ds_list]), dims=('time', 'yh', 'xh'))
ice_rain = xr.DataArray(da.concatenate([da.from_array(ds["/rain_ai_m"]) for ds in ice_ds_list]), dims=('time', 'yh', 'xh'))
ice_sublim = xr.DataArray(da.concatenate([da.from_array(ds["/evap_ai_m"]) for ds in ice_ds_list]), dims=('time', 'yh', 'xh'))
ice_melt = xr.DataArray(da.concatenate([da.from_array(ds["/fresh_ai_m"]) for ds in ice_ds_list]), dims=('time', 'yh', 'xh'))

In [19]:
ice_snow_avg = ice_snow.weighted(areacello.fillna(0)).sum(dim=('yh', 'xh')).compute() * 1000 / (100 * 3600 * 24 * ocn_area)
ice_rain_avg = ice_rain.weighted(areacello.fillna(0)).sum(dim=('yh', 'xh')).compute() * 1000 / (100 * 3600 * 24 * ocn_area)
ice_sublim_avg = ice_sublim.weighted(areacello.fillna(0)).sum(dim=('yh', 'xh')).compute() * 1000 / (100 * 3600 * 24 * ocn_area)
ice_melt_avg = ice_melt.weighted(areacello.fillna(0)).sum(dim=('yh', 'xh')).compute() * 1000 / (100 * 3600 * 24 * ocn_area)

### Compare average fluxes

In [20]:
water_flux_rows = []  # also written to the docs page as a table, below

def print_water_flux_error(flx1, flx2, field_name, time_slice):
    flx1 = flx1.isel(time=time_slice).mean().data
    flx2 = flx2.isel(time=time_slice).mean().data

    print(f'{field_name}: {flx1 * (3600 * 24 * 365)} mm/yr')
    print(f'{field_name} error: {(flx1 - flx2) * (3600 * 24 * 365)} mm/yr')
    print(f'{field_name} error: {(flx1 - flx2) } kg/(m^2s)')
    print(f'{field_name} relative error: {(flx1 - flx2) / flx1}')
    print()
    water_flux_rows.append({
        "Flux": field_name,
        "Mean (mm/yr)": float(flx1 * (3600 * 24 * 365)),
        "Error (mm/yr)": float((flx1 - flx2) * (3600 * 24 * 365)),
        "Error (kg/(m^2s))": float(flx1 - flx2),
        "Relative error": float((flx1 - flx2) / flx1),
    })

In [21]:
time_slice = slice(0, 480)
print_water_flux_error(atm_river_avg, ocn_river_avg, "River runoff", time_slice)
print_water_flux_error(atm_evap_avg, -ocn_evap_avg, "Evaporation", time_slice)
print_water_flux_error(atm_rain_avg, ocn_rain_avg + ice_rain_avg, "Rain", time_slice)
print_water_flux_error(atm_snow_avg, ocn_snow_avg + ice_snow_avg, "Snow", time_slice)
print_water_flux_error(ice_melt_avg, ocn_melt_avg, "Melt water", time_slice)
print_water_flux_error(atm_sublim_avg, -ice_sublim_avg, "Sublimination", time_slice)

River runoff: 156.46643124949244 mm/yr
River runoff error: 15.412965575394844 mm/yr
River runoff error: 4.88741932248695e-07 kg/(m^2s)
River runoff relative error: 0.09850653237446318

Evaporation: 1369.2769794987232 mm/yr
Evaporation error: 0.03724649271677764 mm/yr
Evaporation error: 1.1810785361738218e-09 kg/(m^2s)
Evaporation relative error: 2.720157665282094e-05

Rain: 1126.7194093550795 mm/yr
Rain error: 0.031093898816341826 mm/yr
Rain error: 9.859810634304232e-10 kg/(m^2s)
Rain relative error: 2.759684315204936e-05

Snow: 79.75851318477629 mm/yr
Snow error: 0.0033689192372044477 mm/yr
Snow error: 1.068277282218559e-10 kg/(m^2s)
Snow relative error: 4.2238992462154896e-05

Melt water: 18.066650481587565 mm/yr
Melt water error: 0.00018527911416953532 mm/yr
Melt water error: 5.875162169252135e-12 kg/(m^2s)
Melt water relative error: 1.0255310709550757e-05

Sublimination: 2.3468060834739592 mm/yr
Sublimination error: -0.06444770231819466 mm/yr
Sublimination error: -2.043623234341535

In [22]:
import pandas as pd

water_flux_table = pd.DataFrame(water_flux_rows)
mkmd.table("Global-mean water flux errors between components",
           water_flux_table.to_markdown(index=False, floatfmt=".3g").split("\n"))
water_flux_table

Adding entry to per-notebook markdown: /g/data/tm70/cyb561/repos/get_cm3website-gogo/access-cm3-paper-1/notebooks/polished-python/mkfigs_output_cm3-run-27-07-2026-PD-control/mkmd/flux-conservation.md


Lines appended to /g/data/tm70/cyb561/repos/get_cm3website-gogo/access-cm3-paper-1/notebooks/polished-python/mkfigs_output_cm3-run-27-07-2026-PD-control/mkmd/flux-conservation.md successfully.


,Flux,Mean (mm/yr),Error (mm/yr),Error (kg/(m^2s)),Relative error
0,River runoff,156.466431,15.412966,4.887419e-07,0.098507
1,Evaporation,1369.276979,0.037246,1.181079e-09,0.000027
2,Rain,1126.719409,0.031094,9.859811e-10,0.000028
3,Snow,79.758513,0.003369,1.068277e-10,0.000042
4,Melt water,18.066650,0.000185,5.875162e-12,0.000010
5,Sublimination,2.346806,-0.064448,-2.043623e-09,-0.027462


## Energy flux error

### Atmosphere ocean fluxes

In [23]:
energy_flux_rows = []  # also written to the docs page as a table, below

def print_energy_flux_error(flx1, flx2, field_name, time_slice):
    flx1 = flx1.isel(time=time_slice).mean().data
    flx2 = flx2.isel(time=time_slice).mean().data

    print(f'{field_name}: {flx1} W/m^2')
    print(f'{field_name} error: {(flx1 - flx2) } W/m^2')
    print(f'{field_name} relative error: {(flx1 - flx2) / flx1}')
    print()
    energy_flux_rows.append({
        "Flux": field_name,
        "Mean (W/m^2)": float(flx1),
        "Error (W/m^2)": float(flx1 - flx2),
        "Relative error": float((flx1 - flx2) / flx1),
    })

In [24]:
ocn_lw = datastore.search(variable="rlntds", frequency="1mon").to_dask(xarray_open_kwargs=xr_kw).rlntds
ocn_sw = datastore.search(variable="rsntds", frequency="1mon").to_dask(xarray_open_kwargs=xr_kw).rsntds
ocn_sens = datastore.search(variable="hfsso", frequency="1mon").to_dask(xarray_open_kwargs=xr_kw).hfsso

In [25]:
ocn_lw_avg = ocn_lw.weighted(areacello.fillna(0)).sum(dim=('yh', 'xh')).compute() / earth_area
ocn_sw_avg = ocn_sw.weighted(areacello.fillna(0)).sum(dim=('yh', 'xh')).compute() / earth_area
ocn_sens_avg = ocn_sens.weighted(areacello.fillna(0)).sum(dim=('yh', 'xh')).compute() / earth_area

In [26]:
atm_lw = xr.DataArray(da.concatenate([da.from_array(ds["/fld_s02i203"]) for ds in atm_ds_list]), dims=('time', 'lat', 'lon'))
atm_sw = xr.DataArray(da.concatenate([da.from_array(ds["/fld_s01i203"]) for ds in atm_ds_list]), dims=('time', 'lat', 'lon'))
atm_sens = xr.DataArray(da.concatenate([da.from_array(ds["/fld_s03i228"]) for ds in atm_ds_list]), dims=('time', 'lat', 'lon'))

In [27]:
atm_lw_avg = atm_lw.weighted(ocn_frac * areacella).sum(dim=("lat", "lon")).compute() / earth_area
atm_sw_avg = atm_sw.weighted(ocn_frac * areacella).sum(dim=("lat", "lon")).compute() / earth_area
atm_sens_avg = atm_sens.weighted(ocn_frac * areacella).sum(dim=("lat", "lon")).compute() / earth_area

In [28]:
print_energy_flux_error(atm_lw_avg, ocn_lw_avg, "Longwave", time_slice)
print_energy_flux_error(atm_sw_avg, ocn_sw_avg, "Shortwave", time_slice)
print_energy_flux_error(atm_sens_avg, -ocn_sens_avg, "Sensible heat", time_slice)

Longwave: -35.35889978139189 W/m^2
Longwave error: -0.0011187174491027463 W/m^2
Longwave relative error: 3.1638921347080116e-05

Shortwave: 123.77657242010791 W/m^2
Shortwave error: 0.0037449047068776053 W/m^2
Shortwave relative error: 3.0255359585876148e-05

Sensible heat: 8.573819032684415 W/m^2
Sensible heat error: 0.00025677069823970555 W/m^2
Sensible heat relative error: 2.9948229285090482e-05



### Atmosphere ice fluxes

In [29]:
atm_topmelt = xr.DataArray(da.concatenate([da.from_array(ds["/fld_s03i257"]) for ds in atm_ds_list]), dims=('time', 'pseudo_level_0', 'lat', 'lon'))
atm_botmelt = xr.DataArray(da.concatenate([da.from_array(ds["/fld_s03i510"]) for ds in atm_ds_list]), dims=('time', 'pseudo_level_0', 'lat', 'lon'))

In [30]:
atm_topmelt_avg = atm_topmelt.weighted(ocn_frac * areacella).sum(dim=("pseudo_level_0", "lat", "lon")).compute() / earth_area
atm_botmelt_avg = atm_botmelt.weighted(ocn_frac * areacella).sum(dim=("pseudo_level_0", "lat", "lon")).compute() / earth_area

In [31]:
ice_topmelt = xr.DataArray(da.concatenate([da.from_array(ds["/fsurfn_ai_m"]) for ds in ice_ds_list]), dims=('time', 'nc', 'yh', 'xh'))
ice_botmelt = xr.DataArray(da.concatenate([da.from_array(ds["/fcondtop_ai_m"]) for ds in ice_ds_list]), dims=('time', 'yh', 'xh'))

In [32]:
ice_topmelt_avg = ice_topmelt.weighted(areacello.fillna(0)).sum(dim=('nc', 'yh', 'xh')).compute() / earth_area
ice_botmelt_avg = ice_botmelt.weighted(areacello.fillna(0)).sum(dim=('yh', 'xh')).compute() / earth_area

In [33]:
print_energy_flux_error(atm_topmelt_avg + atm_botmelt_avg, ice_topmelt_avg, "Sea-ice conductive heat flux", time_slice)
print_energy_flux_error(atm_botmelt_avg, ice_botmelt_avg, "Sea-ice conductive heat flux", time_slice)

Sea-ice conductive heat flux: -0.5985597594605955 W/m^2
Sea-ice conductive heat flux error: 0.0010876962236954224 W/m^2
Sea-ice conductive heat flux relative error: -0.0018171890216536148

Sea-ice conductive heat flux: -0.8875936917492312 W/m^2
Sea-ice conductive heat flux error: -0.05348025153615288 W/m^2
Sea-ice conductive heat flux relative error: 0.06025307754357325



In [34]:
energy_flux_table = pd.DataFrame(energy_flux_rows)
mkmd.table("Global-mean energy flux errors between components",
           energy_flux_table.to_markdown(index=False, floatfmt=".3g").split("\n"))
energy_flux_table

Adding entry to per-notebook markdown: /g/data/tm70/cyb561/repos/get_cm3website-gogo/access-cm3-paper-1/notebooks/polished-python/mkfigs_output_cm3-run-27-07-2026-PD-control/mkmd/flux-conservation.md
Lines appended to /g/data/tm70/cyb561/repos/get_cm3website-gogo/access-cm3-paper-1/notebooks/polished-python/mkfigs_output_cm3-run-27-07-2026-PD-control/mkmd/flux-conservation.md successfully.


,Flux,Mean (W/m^2),Error (W/m^2),Relative error
0,Longwave,-35.358900,-0.001119,0.000032
1,Shortwave,123.776572,0.003745,0.000030
2,Sensible heat,8.573819,0.000257,0.000030
3,Sea-ice conductive heat flux,-0.598560,0.001088,-0.001817
4,Sea-ice conductive heat flux,-0.887594,-0.053480,0.060253
